In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
shipments_bronze_path = f"{BRONZE_PATH}/shipments"
shipments_silver_path = f"{SILVER_PATH}/shipments"

In [0]:
df_shipments_bronze = spark.read.format("delta") \
    .load(shipments_bronze_path)

In [0]:
display(df_shipments_bronze.limit(20))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
windowSpec = Window.partitionBy("shipment_id").orderBy(F.col("updated_at").desc())

df_shipments_silver = df_shipments_bronze \
    .withColumn(
        "rn",
        F.row_number().over(windowSpec)
    ) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

In [0]:
%skip
df_shipments_silver.write.format("delta").mode("append").save(shipments_silver_path)

In [0]:
from delta.tables import DeltaTable

shipments_silver_table = DeltaTable.forPath(
    spark,
    shipments_silver_path
)

shipments_silver_table.alias("target") \
    .merge(
        df_shipments_silver.alias("source"),
        "source.shipment_id = target.shipment_id"
    ) \
    .whenMatchedUpdate(
        set = {
            "order_id": "source.order_id",
            "warehouse_id": "source.warehouse_id",
            "shipment_date": "source.shipment_date",
            "delivery_date": "source.delivery_date",
            "shipment_status": "source.shipment_status",
            "updated_at": "source.updated_at"
        }
    ) \
    .whenNotMatchedInsertAll() \
    .execute()